# Fase 5: RUL Estimation - Deep Learning (LSTM & GRU)

## Tujuan
Memprediksi Remaining Useful Life (RUL) mesin dalam satuan hari/jam menggunakan arsitektur LSTM.

## Langkah-langkah
- 5.1: Rekayasa Target RUL (Hitung mundur dari titik failure, piece-wise cap 30 hari)
- 5.2: Transformasi Data ke Tensor 3D (Sliding Window, time_steps=24)
- 5.3: Arsitektur LSTM
- 5.4: Training dengan Early Stopping
- 5.5: Evaluasi Regresi (MAE, RMSE) & Visualisasi RUL Aktual vs Prediksi
- 5.6: Ekspor Model Pemenang (rul_lstm.h5 / rul_gru.h5)

### Eksperimen 1: Rekayasa Target RUL (Remaining Useful Life)
Langkah awal untuk mengubah data klasifikasi/anomali menjadi data regresi (angka hari). Kita membalikkan waktu `timestamp` kerusakan menggunakan teknik *Piece-wise RUL* maksimal 30 hari.


In [ ]:
import pandas as pd
import numpy as np

# 1. Muat data hasil tahap Feature Engineering
df = pd.read_csv('../data/processed/sensor_features_engineered.csv')

# 2. Pastikan kolom timestamp berformat datetime (belum menjadi index)
df['timestamp'] = pd.to_datetime(df['timestamp'])

# 3. Cari waktu maksimum (terakhir) saat mesin mati (failure == 1) untuk tiap mesin
# Buat referensi tabel terpisah menggunakan groupby, kemudian gabungkan
max_failure_times = df[df['failure'] == 1].groupby('machine_id')['timestamp'].max().reset_index()
max_failure_times = max_failure_times.rename(columns={'timestamp': 'max_failure_time'})

# Gabungkan max_failure_time ke Dataframe utama menggunakan left join
df = df.merge(max_failure_times, on='machine_id', how='left')

# 4. Hitung mundur Sisa Umur (RUL_days) 
# Menghitung selisih waktu dalam hari (menggunakan pd.Timedelta agar mendapatkan nilai desimal/float hari)
df['RUL_days'] = (df['max_failure_time'] - df['timestamp']) / pd.Timedelta(days=1)

# (Opsional tapi penting): Filter nilai negatif dari sensor anomali yang masih menyala jika mesin sudah diset mati
df = df[df['RUL_days'] >= 0]

# 5. Terapkan Trik Senior (Piece-wise RUL) limit maksimal 30 hari
# Potong semua nilai yang lebih dari 30 menjadi mentok 30 saja
df['RUL_days'] = df['RUL_days'].clip(upper=30)

# 6. Hapus baris yang berstatus NaN di kolom RUL_days (mesin yang belum pernah terdeteksi rusak)
df = df.dropna(subset=['RUL_days'])

# 7. Bersihkan kolom yang sudah tidak terpakai lagi
df = df.drop(columns=['failure', 'max_failure_time'])

# 8. Kembalikan timestamp menjadi index DataFrame
df = df.set_index('timestamp')

# 9. Cetak distribusi statistik akhir untuk memvalidasi Piece-wise RUL
print("=== Distribusi Target RUL_days (Piece-wise 30 Hari) ===")
print(df['RUL_days'].describe())


### Eksperimen 2: Transformasi Data 2D ke Tensor 3D (Hukum Anti-Data Leakage)
Membelah data (`train_test_split`) tanpa acak (`shuffle=False`), lalu memotong data menggunakan *Sliding Window* sepanjang 24 baris waktu ke belakang.


In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Pisahkan fitur X (hapus kolom RUL_days dan machine_id) dan target y (RUL_days)
X = df.drop(columns=['RUL_days', 'machine_id'], errors='ignore')
y = df['RUL_days']

# 2. Hukum Split Waktu: train_test_split dengan shuffle=False (Linear Split) 70:30
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

# 3. Fungsi memotong data ke Tensor 3D sesuai Hukum Anti-Kebocoran Waktu
def create_sequences(X, y, time_steps=24):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X.iloc[i:(i + time_steps)].values)
        ys.append(y.iloc[i + time_steps])
    return np.array(Xs), np.array(ys)

# 4. Terapkan fungsi pada data Latih
time_steps = 24
X_train_3D, y_train_seq = create_sequences(X_train, y_train, time_steps)

# 5. Terapkan fungsi pada data Ujian
X_test_3D, y_test_seq = create_sequences(X_test, y_test, time_steps)

# 6. Cetak dimensi (shape) akhir Tensor 3D
print("Shape X_train_3D:", X_train_3D.shape)
print("Shape X_test_3D:", X_test_3D.shape)


### Eksperimen 3: Membangun Kerangka Arsitektur LSTM (Vanilla)
Inisialisasi model Artificial Neural Network `Sequential` menggunakan dua tumpukan fungsi memori jangka panjang bersyarat (LSTM).


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense

# 1 & 2. Inisialisasi model
model_lstm = Sequential()

# 3. Tambahkan lapisan input memori pertama
# Catatan: Menggunakan index [1] dan [2] untuk mengambil (time_steps, features) dari X_train_3D
model_lstm.add(LSTM(units=64, return_sequences=True, input_shape=(X_train_3D.shape[1], X_train_3D.shape[2])))

# 4. Tambahkan Dropout untuk mencegah overfitting
model_lstm.add(Dropout(0.2))

# 5. Tambahkan lapisan memori kedua
model_lstm.add(LSTM(units=32, return_sequences=False))

# 6. HUKUM MUTLAK LAYER OUTPUT: Dense(1) tanpa aktivasi (Kasus Regresi)
model_lstm.add(Dense(units=1))

# 7. Cetak kerangka arsitektur
model_lstm.summary()


### Eksperimen 4: Training Model dengan Rem Darurat (Early Stopping)
Menjalankan kompilasi menggunakan Metrik `MSE` (Mean Squared Error) dan `MAE` (Mean Absolute Error). 
*(Catatan: Ini akan gagal / Error karena sensor belum di-scale dan tercampur NaN).*


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# 1. Compile model LSTM (Regresi: Mean Squared Error atau Mean Absolute Error)
model_lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])

# 2 & 3. Konfigurasi Rem Darurat (Early Stopping)
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# 4. Latih model dengan tensor 3D dan label sequence y_train_seq
history = model_lstm.fit(
    X_train_3D, y_train_seq,
    epochs=50,
    batch_size=32,
    validation_data=(X_test_3D, y_test_seq),
    callbacks=[early_stop]
)


### Eksperimen 5: Diagnosa Prediksi Awal via Plotting Aktual (Garis)
Visualisasi hasil regresi (jika kode loss tidak Error/NaN). Digunakan untuk melihat seberapa jauh kurva algoritma menempel pada garis hitung mundur mesin.


In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

# Fungsi pemotong Tensor 3D (dideklarasikan ulang untuk berjaga-jaga)
def create_sequences(X, y, time_steps=24):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        # Ambil jendela waktu untuk fitur X
        Xs.append(X[i:(i + time_steps)])
        # Ambil 1 nilai target y di ujung jendela waktu
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

# 1. & 2. Terapkan Linear Split (Haram Mengacak)
# Asumsi DataFrame df, X, dan y sudah terdefenisi di memori dari blok awal Anda
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

# 3. Terapkan Hukum Normalisasi (Sangat Penting)
scaler = MinMaxScaler()
# Wajib: Fit & Transform hanya pada data latih agar AI tidak menyontek distribusi masa depan
X_train_scaled = scaler.fit_transform(X_train)
# Wajib: Hanya transform pada data ujian berdasarkan ilmu dari data latih
X_test_scaled = scaler.transform(X_test)

# 4. Bentuk ulang Tensor 3D menggunakan data yang SUDAH DI-SCALE!
# Perhatikan kita menggunakan .values pada y agar selaras dengan input array X_scaled
time_steps = 24
X_train_3D, y_train_seq = create_sequences(X_train_scaled, y_train.values)
X_test_3D, y_test_seq = create_sequences(X_test_scaled, y_test.values)

print(f"Data Telah Ternormalisasi! Shape X_train_3D Baru: {X_train_3D.shape}")

# ==========================================================
# RESET OTAK AI (Re-inisialisasi agar beban NaN lama terhapus)
# ==========================================================

# 5. Bangun Ulang Arsitektur LSTM (Sama persis instruksinya)
model_lstm = Sequential()
model_lstm.add(LSTM(units=64, return_sequences=True, input_shape=(X_train_3D.shape[1], X_train_3D.shape[2])))
model_lstm.add(Dropout(0.2))
model_lstm.add(LSTM(units=32, return_sequences=False))
model_lstm.add(Dense(units=1))

# 6. Compile Ulang Model
model_lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Siapkan Early Stopping (Rem Darurat)
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# 7. Mulai Latih Ulang dengan Tensor yang Sehat (Sudah Scaled)
print("Memulai Pelatihan Model LSTM yang Telah Dinormalisasi...")
history = model_lstm.fit(
    X_train_3D, y_train_seq,
    epochs=50,
    batch_size=32,
    validation_data=(X_test_3D, y_test_seq),
    callbacks=[early_stop]
)


### Revisi 1: Menyembuhkan Exploding Gradient (MinMaxScaler)
Melakukan injeksi `MinMaxScaler` pada data Latih (secara mutlak) dan merefleksikannya pada data Ujian agar seluruh deret tensor seragam melenggang ke rentang 0.0 - 1.0.


In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

# --- 1. & 2. SAPU BERSIH NAN & TEKS ---
df.dropna(inplace=True)

# SOLUSI: Buang 'RUL_days' DAN 'machine_id' agar X murni berisi angka sensor mutlak
X = df.drop(columns=['RUL_days', 'machine_id'], errors='ignore') 
y = df['RUL_days']

# --- 3. LINEAR SPLIT HARAM ACAK ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

# --- 4. HUKUM NORMALISASI FITUR ---
scaler = MinMaxScaler()
# Sekarang tidak akan error karena X murni angka
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 5. BENTUK TENSOR 3D (ANTI-BOCOR WAKTU) ---
def create_sequences(X, y, time_steps=24):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

time_steps = 24
X_train_3D, y_train_seq = create_sequences(X_train_scaled, y_train.values)
X_test_3D, y_test_seq = create_sequences(X_test_scaled, y_test.values)

print(f"✅ Sapu Bersih Selesai! Data bebas NaN dan murni angka.")
print(f"✅ Shape X_train_3D Baru: {X_train_3D.shape}")

# ==========================================================
# --- 6. & 7. BANGUN ULANG ARSITEKTUR ---
# ==========================================================
model_lstm = Sequential()
model_lstm.add(Input(shape=(X_train_3D.shape[1], X_train_3D.shape[2])))

# LSTM Memori #1
model_lstm.add(LSTM(units=64, return_sequences=True))
model_lstm.add(Dropout(0.2))

# LSTM Memori #2
model_lstm.add(LSTM(units=32, return_sequences=False))

# Dense Layer Regresi Mutlak
model_lstm.add(Dense(units=1))

# --- 8. & 9. COMPILE DAN LATIH DENGAN REM DARURAT ---
model_lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("🚀 Memulai Pelatihan Model LSTM...")
history = model_lstm.fit(
    X_train_3D, y_train_seq,
    epochs=50,
    batch_size=32,
    validation_data=(X_test_3D, y_test_seq),
    callbacks=[early_stop]
)


### Revisi 2: Standarisasi Keras 3.x dan Sanitasi DataFrame Asli
Mereset memori kernel agar bebas dari sisa baris `NaN` (*Rolling Means* bocor) menggunakan `df.dropna()`. Menyelipkan `Input()` di *Sequential* model untuk mengakhiri peringatan merah Keras versi teranyar.


In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping

# 1 & 2. Bangun Arsitektur LSTM yang Lebih "Deep" dan Kebal Overfit
model_lstm_tuned = Sequential()

# Layer Pintu Masuk Data (Input)
model_lstm_tuned.add(Input(shape=(X_train_3D.shape[1], X_train_3D.shape[2])))

# Layer Memori Utama (Diperbesar Kapasitas Kapal)
model_lstm_tuned.add(LSTM(units=128, return_sequences=True))
# Dropout super agresif 30% (Untuk memaksa AI tidak sok tahu menghafal masa lalu)
model_lstm_tuned.add(Dropout(0.3))

# Layer Memori Sekunder
model_lstm_tuned.add(LSTM(units=64, return_sequences=False))
model_lstm_tuned.add(Dropout(0.2))

# Layer Logika Ekstra (Penjembatan Memori dengan Regresi Angka Linear)
model_lstm_tuned.add(Dense(units=32, activation='relu'))

# Layer Output Mutlak
model_lstm_tuned.add(Dense(units=1))

# 3. Custom Optimizer (Melambatkan Kecepatan Belajar AI secara Sengaja)
# Default adam adalah 0.001, kita potong setengahnya!
custom_adam = Adam(learning_rate=0.0005)

# 4. Compile Model dengan Senjata Baru
model_lstm_tuned.compile(optimizer=custom_adam, loss='mse', metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# 5. Eksekusi Pelatihan Eksperimen #2
print("🚀 Memulai Pelatihan TUNING - LSTM Deep Architecture...")
history_tuned = model_lstm_tuned.fit(
    X_train_3D, y_train_seq,
    epochs=50,
    batch_size=32,
    validation_data=(X_test_3D, y_test_seq),
    callbacks=[early_stop]
)


### Eksekusi Tuning: Ekspansi Kedalaman LSTM (Deep Architecture)
Mematahkan nilai MAE yang stagnan di angka belasan dengan memasang *Learning Rate* buatan (Adam=0.0005) agar model lebih rileks menuruni gradien Loss, menambah ketebalan *Dropout* menjadi `0.3`, serta menyisipkan `Dense(32, Relu)` jembatan.


In [ ]:
import matplotlib.pyplot as plt
import os

# --- LANGKAH 5.5: VISUALISASI AKTUAL VS PREDIKSI ---
# 1. Tebak RUL menggunakan model yang sudah di-tuning (LSTM) pada data tersembunyi Ujian
y_pred_tuned = model_lstm_tuned.predict(X_test_3D)

# 3. Siapkan Kanvas Grafik berukuran 14x6 (Standard Analyst)
plt.figure(figsize=(14, 6))

# 4. Plot Garis: 200 Titik Pertama
# (Kita batasi 200 agar grafiknya terlihat bentuk tangga "Hitung Mundurnya" secara jelas)
plt.plot(y_test_seq[:200], label='RUL Aktual Asli', color='blue', linewidth=2)
plt.plot(y_pred_tuned[:200], label='Prediksi LSTM Tuned', color='red', linestyle='dashed')

# 5. Kostumisasi Judul dan Label
plt.title('Visualisasi RUL Aktual vs Prediksi LSTM Tuned (200 Jam Pertama)', fontsize=14)
plt.xlabel('Waktu (Jam)', fontsize=12)
plt.ylabel('Sisa Umur (Hari)', fontsize=12)

# Mengaktifkan latar Grid dan penanda garis (Legend)
plt.legend()
plt.grid(True)
plt.show()

# --- LANGKAH 5.6: EKSPOR PEMENANG (MODEL PENYIMPANAN) ---
# Memastikan direktori `models` ada (Aman)
if not os.path.exists('../models'):
    os.makedirs('../models')

# 6. Simpan keseluruhan "Otak" AI (Arsitektur + Bobot Memory)
model_lstm_tuned.save('../models/rul_lstm.h5')

# 7. Konfirmasi Selesai
print("\n✅ Model Peramal Waktu (LSTM) berhasil diekspor ke '../models/rul_lstm.h5'")
print("🎉 FASE 5 RESMI SELESAI!")


### Finalisasi (Deliverable Role A)
Visualisasi terakhir model yang ter-Tuning dan proses Serialization (ekspor permanen file biner arsitektur AI Anda ke dalam `.h5`) untuk dijemput oleh *Environment Backend*.


In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

# --- 1. Awal DataFrame Bersih ---
# (Pastikan kolom rujukan belum terhapus akibat kode sel lama)
df.dropna(inplace=True)

# --- 2. Linear Split UTUH (Terhadap DataFrame Induk) ---
split_idx = int(len(df) * 0.7)
df_train = df.iloc[:split_idx].copy()
df_test = df.iloc[split_idx:].copy()

# --- 3. Definisikan Kolom Fitur Murni ---
fitur_sensor = [col for col in df.columns if col not in ['machine_id', 'RUL_days']]

# --- 4. HUKUM NORMALISASI YANG BENAR ---
scaler = MinMaxScaler()
# Fit & Transform hanya di df_train murni
df_train[fitur_sensor] = scaler.fit_transform(df_train[fitur_sensor])
# Transform saja di df_test
df_test[fitur_sensor] = scaler.transform(df_test[fitur_sensor])

# --- 5. FUNGSI PEMBUAT URUTAN WAKTU PER-MESIN (Sangat Krusial) ---
def create_sequences_per_machine(df_data, time_steps=24):
    Xs, ys = [], []
    # Loop Grouping agar data Mesin A tidak nyambung ke awal Mesin B
    for machine, group in df_data.groupby('machine_id'):
        X_group = group[fitur_sensor].values
        y_group = group['RUL_days'].values
        
        # Potong window HANYA jika baris data grup ini cukup panjang
        for i in range(len(X_group) - time_steps):
            Xs.append(X_group[i:(i + time_steps)])
            ys.append(y_group[i + time_steps])
            
    return np.array(Xs), np.array(ys)

time_steps = 24
print("⏳ Sedang memuat pemotongan 3D Per-Mesin (Bisa memakan waktu beberapa detik)...")
X_train_3D, y_train_seq = create_sequences_per_machine(df_train, time_steps)
X_test_3D, y_test_seq = create_sequences_per_machine(df_test, time_steps)

print(f"✅ Data 3D Berhasil Digabung Aman!")
print(f"Shape X_train_3D: {X_train_3D.shape}")

# --- 6. BANGUN & LATIH ULANG MODEL FINAL YANG SEHAT ---
model_lstm_final = Sequential()
model_lstm_final.add(Input(shape=(X_train_3D.shape[1], X_train_3D.shape[2])))
model_lstm_final.add(LSTM(units=64)) # Ringkas tanpa sequence sambung
model_lstm_final.add(Dense(units=1))

model_lstm_final.compile(optimizer='adam', loss='mse', metrics=['mae'])
early_stop_final = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("\n🚀 Memulai Pelatihan Model LSTM Cepat (Revisi Final)...")
history_final = model_lstm_final.fit(
    X_train_3D, y_train_seq,
    epochs=15,          # Diturunkan sesuai instruksi agar cepat
    batch_size=32,
    validation_data=(X_test_3D, y_test_seq),
    callbacks=[early_stop_final]
)
